# دليل عملي لمكتبة arabic_nlp (عربي + مصري)
هذا الدفتر يعرض اغلب واجهات المكتبة: التطبيع، التقطيع، الستمنج،
ازالة الكلمات الشائعة، خط الانابيب، ميزات الاسترجاع، وتمثيلات النص.
الامثلة تشمل العربية الفصحى والمصرية.

> ملاحظة: تمثيلات Word2Vec/BERT/ELMo تحتاج حزم اختيارية.
> كذلك ادوات الاسترجاع والتمثيلات مستوردة من arabic_nlp.ir و arabic_nlp.representations.
> متغير RUN_HEAVY يتحكم في تشغيل النماذج الثقيلة.

In [8]:
import unicodedata

import arabic_nlp as rna
from arabic_nlp import (
    normalize_arabic,
    soundex_arabic,
    soundex_english,
    thesaurus_lookup,
    lemmatize_arabic,
    lemmatize_english,
    tokenize_arabic,
    tokenize_sentences,
    tokenize_mixed,
    split_clitic_tokens,
    get_arabic_stopwords,
    load_stopwords,
    remove_stopwords,
    light_stem,
    stem_tokens,
    get_isri_stemmer,
    get_porter_stemmer,
    get_snowball_stemmer,
    ArabicPreprocessor,
    PreprocessConfig,
    preprocess_text,
 )
from arabic_nlp.ir import (
    term_frequencies,
    document_frequencies,
    inverse_document_frequencies,
    word_ngrams,
    char_ngrams,
    document_stats,
    CountVectorizer,
    TfidfVectorizer,
    BM25Vectorizer,
    HashingVectorizer,
 )
from arabic_nlp.representations import (
    CBOW,
    SkipGram,
    Word2VecConfig,
    BertEncoder,
    ElmoEncoder,
 )

# Backward-compatible fallback for older installs missing case_fold.
if hasattr(rna, "case_fold"):
    case_fold = rna.case_fold
else:
    def case_fold(text: str, language: str = "arabic") -> str:
        if language == "arabic":
            return normalize_arabic(text)
        return unicodedata.normalize("NFKC", text).casefold()

from pathlib import Path
import tempfile

text_msa = "ذهب الولد الى المدرسة صباحا وكان الجو جميلا."
text_egy = "الولد راح المدرسة النهاردة والجو حلو قوي."
text_mixed = "ده phone جديد ومفيد"

RUN_HEAVY = False

ImportError: cannot import name 'case_fold' from 'arabic_nlp.normalization' (c:\Users\yufss\AppData\Local\Programs\Python\Python311\Lib\site-packages\arabic_nlp\normalization.py)

## التطبيع وطي الحالة وSoundex

In [ ]:
text_with_diacritics = "ـالمَدرَسَةُ"
print("normalize_arabic:", normalize_arabic(text_with_diacritics))
print("case_fold arabic:", case_fold("أَحْمَد", language="arabic"))
print("case_fold english:", case_fold("Hello WORLD", language="english"))

print("soundex_arabic:", soundex_arabic("كتبت"))
print("soundex_english:", soundex_english("Robert"))

## القاموس المصري واللمتة

In [ ]:
try:
    print("thesaurus (arabic):", thesaurus_lookup("ترك", source="arabic")[:10])
    print("thesaurus (english):", thesaurus_lookup("school", source="english")[:10])
    print("lemmatize_arabic:", lemmatize_arabic("مدارس"))
    print("lemmatize_arabic (مصري):", lemmatize_arabic("عايزين"))
    print("lemmatize_english:", lemmatize_english("running"))
except Exception as exc:
    print("تعذر تشغيل القاموس/التلميد:", exc)

## التقطيع (Tokenization)

In [ ]:
print("tokenize_arabic:", tokenize_arabic(text_msa))
print("tokenize_arabic + split_clitics:", tokenize_arabic("والكتاب الجديد", split_clitics=True))
tokens = tokenize_arabic("وللمدرسة")
print("split_clitic_tokens:", split_clitic_tokens(tokens))

sentences = tokenize_sentences("الجو جميل. هل تحب الطقس؟ انا احبه.")
print("tokenize_sentences:", sentences)

print("tokenize_mixed:", tokenize_mixed(text_mixed))

## الكلمات الشائعة (Stopwords)

In [ ]:
tokens_egy = tokenize_arabic(text_egy)
sw = get_arabic_stopwords()
print("قبل:", tokens_egy)
print("بعد:", remove_stopwords(tokens_egy, stopwords=sw))

with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "custom_stopwords.txt"
    path.write_text("الولد\nالجو\n", encoding="utf-8")
    custom_sw = load_stopwords(str(path))
    print("مخصصة:", remove_stopwords(tokens_egy, stopwords=custom_sw))

## الستمنج (Stemming)

In [ ]:
tokens = tokenize_arabic("والمدارس الجميلة")
print("light_stem:", [light_stem(t) for t in tokens])
print("stem_tokens:", stem_tokens(tokens))

try:
    isri = get_isri_stemmer()
    print("isri:", [isri(t) for t in tokens])
except RuntimeError as exc:
    print("ISRI غير متاح:", exc)

try:
    porter = get_porter_stemmer()
    snowball = get_snowball_stemmer("english")
    print("porter:", porter("running"))
    print("snowball:", snowball("generously"))
except RuntimeError as exc:
    print("NLTK غير متاح:", exc)

## خط الانابيب (Preprocessing Pipeline)

In [ ]:
print("preprocess_text MSA:", preprocess_text(text_msa))
print("preprocess_text Egyptian:", preprocess_text(text_egy, split_clitics=True))

config = PreprocessConfig(split_clitics=True, use_lemmatizer=True)
pre = ArabicPreprocessor.from_config(config)
try:
    print("ArabicPreprocessor:", pre(text_egy))
except Exception as exc:
    print("تعذر تشغيل اللمتة القاموسية:", exc)

try:
    isri = get_isri_stemmer()
    pre_isri = ArabicPreprocessor(stemmer=isri)
    print("ArabicPreprocessor (ISRI):", pre_isri(text_msa))
except RuntimeError as exc:
    print("ISRI غير متاح:", exc)

tokens_msa = preprocess_text(text_msa)
tokens_egy = preprocess_text(text_egy)
corpus_tokens = [tokens_msa, tokens_egy]

## احصائيات و n-grams

In [ ]:
doc_freqs = document_frequencies(corpus_tokens)
idf = inverse_document_frequencies(doc_freqs, doc_count=len(corpus_tokens))

print("term_frequencies:", term_frequencies(tokens_msa))
print("document_frequencies:", doc_freqs)
print("inverse_document_frequencies:", idf)

print("word_ngrams (1-2):", word_ngrams(tokens_msa, ngram_range=(1, 2)))
print("char_ngrams (3-4):", char_ngrams(text_egy, ngram_range=(3, 4))[:10])
print("document_stats:", document_stats(tokens_egy))

## تمثيل النص للبحث (Count/TF-IDF/BM25/Hashing)

In [ ]:
count_vec = CountVectorizer(ngram_range=(1, 2))
count_vec.fit(corpus_tokens)
print("Count:", count_vec.transform(tokens_egy, return_sparse=True))

tfidf_vec = TfidfVectorizer(ngram_range=(1, 2))
tfidf_vec.fit(corpus_tokens)
print("TF-IDF (dense):", tfidf_vec.transform(tokens_egy, return_sparse=False))

bm25_vec = BM25Vectorizer()
bm25_vec.fit(corpus_tokens)
print("BM25:", bm25_vec.transform(tokens_egy, return_sparse=True))

hash_vec = HashingVectorizer(n_features=32)
print("Hashing:", hash_vec.transform(tokens_egy, return_sparse=True))

## تمثيلات Word2Vec (CBOW و Skip-gram)

In [ ]:
try:
    w2v_config = Word2VecConfig(vector_size=20, window=2, epochs=2, min_count=1)
    cbow = CBOW(w2v_config)
    cbow.fit(corpus_tokens)
    vec = cbow.encode(tokens=tokens_egy, pooling="mean")
    print("CBOW vec len:", len(vec))

    with tempfile.TemporaryDirectory() as tmp:
        cbow.save(tmp)
        cbow_loaded = CBOW.load(tmp)
        print("CBOW loaded vec len:", len(cbow_loaded.encode(tokens=tokens_egy)))

    sg = SkipGram(w2v_config)
    sg.fit(corpus_tokens)
    print("SkipGram vec len:", len(sg.encode(tokens=tokens_msa)))
except RuntimeError as exc:
    print("Word2Vec غير متاح:", exc)

## تمثيل BERT

In [ ]:
if RUN_HEAVY:
    try:
        bert = BertEncoder(model_name="bert-base-multilingual-cased", pooling="cls")
        vec = bert.encode(text=text_msa)
        print("BERT vec len:", len(vec))
    except RuntimeError as exc:
        print("BERT غير متاح:", exc)
else:
    print("BERT متوقف افتراضيا. اجعل RUN_HEAVY = True لتشغيله.")

## تمثيل ELMo

In [ ]:
options_file = "path/to/options.json"
weight_file = "path/to/weights.hdf5"

if RUN_HEAVY and Path(options_file).exists() and Path(weight_file).exists():
    try:
        elmo = ElmoEncoder(
            options_file=options_file,
            weight_file=weight_file,
            pooling="mean",
        )
        vec = elmo.encode(text=text_egy)
        print("ELMo vec len:", len(vec))
    except RuntimeError as exc:
        print("ELMo غير متاح:", exc)
else:
    print("ELMo متوقف افتراضيا. عيّن الملفات وشغّل RUN_HEAVY.")